# Optimize spatio-temporal synthetic-data parameters

Goal: find parameters such that IID, cluster-aware, and spatial TOST conclude equivalence at Δ=1, but the spatio-temporal TOST rejects equivalence at Δ=1.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import os, sys, json
from pathlib import Path

# If this notebook is in the directory that contains `main_module/`, add it to sys.path
# ROOT = Path('.').resolve()
# if str(ROOT) not in sys.path:
#     sys.path.insert(0, str(ROOT))

# print('cwd:', ROOT)

from pyTOST.data_gen.optimize_spatiotemporal_params import search
from pyTOST.data_gen.params_io import save_best

In [2]:
best, hist = search(
    seed=123,
    n_iter=1000,
    margin=1.0,
    verbose_every=1,
    # use_subprocess=True,
    candidate_timeout_seconds=300,
    debug_stage_prints=True,
    # debug_timing=True,
    # slow_step_seconds=5,
    # heartbeat_seconds=5,
)

[evaluate] stage=generate
[evaluate] stage=diff
[evaluate] stage=iid
[evaluate] stage=cluster
[evaluate] stage=temporal
[   1/1000] cur eq(iid,clu,tmp,spa,st)=(False,False,False,False,False) cur CI_st=(nan, nan) score=50.37 notes=Screen rejected | best score=50.37 best eq=(False,False,False,False,False) best CI_st=(nan, nan)
[evaluate] stage=generate
[evaluate] stage=diff
[evaluate] stage=iid
[evaluate] stage=cluster
[evaluate] stage=temporal
[   2/1000] cur eq(iid,clu,tmp,spa,st)=(False,False,False,False,False) cur CI_st=(nan, nan) score=54.15 notes=Screen rejected | best score=50.37 best eq=(False,False,False,False,False) best CI_st=(nan, nan)
[evaluate] stage=generate
[evaluate] stage=diff
[evaluate] stage=iid
[evaluate] stage=cluster
[evaluate] stage=temporal
[evaluate] stage=spatial
[evaluate] stage=spatiotemporal
[   3/1000] cur eq(iid,clu,tmp,spa,st)=(True,True,True,True,False) cur CI_st=(0.26443445596963777, 1.4921816967220476) score=0.3922 notes=Full eval | best score=0.3922 b

In [10]:
from pprint import pprint

In [11]:
pprint(best.params)

{'delta': 0.9519510056343431,
 'length_scale': 11.331607149197897,
 'n_space': 29,
 'n_time': 18,
 'obs_sd': 0.14593142813976095,
 'rho': 0.9414443879036649,
 'seed': 10125,
 'spatial_sd': 1.4232993880098288}


In [13]:
print(f'IID: {best.eq_iid}\nCluster: {best.eq_cluster}\nSpatial: {best.eq_spatial}\nTemporal: {best.eq_temporal}\nSpatiotemporal: {best.eq_spatiotemporal}')

IID: True
Cluster: True
Spatial: True
Temporal: True
Spatiotemporal: False


In [14]:
print(f'IID: {best.ci_iid}\nCluster: {best.ci_cluster}\nSpatial: {best.ci_spatial}\nTemporal: {best.ci_temporal}\nSpatiotemporal: {best.ci_spatiotemporal}')

IID: (0.9066510800035088, 0.9715580978167402)
Cluster: (0.9070265833474728, 0.9711825944727762)
Spatial: (0.9066978190399209, 0.9715113590066113)
Temporal: (0.8798010086486208, 0.9984081691716278)
Spatiotemporal: (0.26443445596963777, 1.4921816967220476)


In [15]:
out_path = 'best_spatiotemporal_params.json'
save_best(best, out_path)
print('Wrote', out_path)

Wrote best_spatiotemporal_params.json


In [16]:
import pandas as pd
from pyTOST.engines.iid_tost import IIDTOST
from pyTOST.engines.cluster_tost import ClusterTOST
from pyTOST.engines.spatial_tost import SpatialTOST
from pyTOST.engines.spatiotemporal_tost import SpatioTemporalTOST
from pyTOST.data_gen import synthetic_tost_data

# Rebuild a dataset using the best params and report results at Δ in {1,3,5}
p = dict(best.params)
n_buildings = int(p.pop('n_buildings', 10))

df_long, meta = synthetic_tost_data.generate_spatiotemporal(**{k:v for k,v in p.items() if k!='_meta'})
wide = df_long.pivot(index='sample_id', columns='arm', values='y').reset_index()
meta2 = df_long[df_long['arm']=='A'][['sample_id','x','y_sp','t']].copy()
meta2 = meta2.merge(wide, on='sample_id')
meta2['diff'] = meta2['B'] - meta2['A']

# derive space index and assign buildings by chunking
n_time = int(meta2['t'].max()+1)
space_idx = (meta2['sample_id'] // max(n_time,1)).astype(int)
space_n = int(space_idx.nunique())
n_buildings = max(1, min(n_buildings, space_n))
per = int((space_n + n_buildings - 1) / n_buildings)
meta2['building_id'] = (space_idx // max(per,1)).astype(int)

df = meta2[['sample_id','building_id','x','y_sp','t','diff']].copy()

margins=[1,3,5]
alpha=0.05
res_iid = IIDTOST('diff').fit(df, alpha=alpha, margins=margins)
res_clu = ClusterTOST('diff','building_id').fit(df, alpha=alpha, margins=margins)
res_spa = SpatialTOST('diff','building_id','x','y_sp').fit(df, alpha=alpha, margins=margins)
res_st  = SpatioTemporalTOST('diff','building_id','t','x','y_sp').fit(df, alpha=alpha, margins=margins)

df_results = pd.concat([
    res_iid.assign(engine='IID'),
    res_clu.assign(engine='Cluster'),
    res_spa.assign(engine='Spatial'),
    res_st.assign(engine='SpatioTemporal'),
], ignore_index=True)

In [17]:
df_results.loc[df_results.delta == 1.0][['method', 'engine', 'ci_low', 'ci_high', 'equivalent']]

,method,engine,ci_low,ci_high,equivalent
0,IID OLS (t-CI),IID,0.906651,0.971558,True
3,OLS + cluster-robust SE (df=9),Cluster,0.902661,0.975548,True
6,Matérn GLS (REML) + LR CI,Spatial,0.906698,0.971511,True
9,Joint separable spatiotemporal ML (AR1 ⊗ Matér...,SpatioTemporal,0.264434,1.492182,False
